# Review 1 — Regression Model Comparison

This notebook consolidates the outputs from the ten required regression notebooks.

It creates the common test-set comparison table, identifies the two best models by R², performs 5-fold cross-validation for those models, and generates the required diagnostic plots.

## 1. Imports and Project Paths

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import cross_val_score

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "Review-1" and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from preprocessing import (
    RANDOM_STATE,
    TARGET,
    load_wine_dataset,
    clean_wine_dataset,
    split_data,
    make_preprocessor,
    regression_metrics,
)

DATA_PATH = PROJECT_ROOT / "data" / "winequality-white.csv"
RESULTS_DIR = PROJECT_ROOT / "results" / "regression"
MODELS_DIR = PROJECT_ROOT / "models"

sns.set_theme(style="whitegrid")

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)
print("Results directory:", RESULTS_DIR)
print("Models directory:", MODELS_DIR)

## 2. Load the Same Wine Quality Data and Prepare the Common Split

The comparison uses the same dataset, target, engineered feature, random state, and 80:20 split used by the regression pipeline.

In [ ]:
df = load_wine_dataset(DATA_PATH)
df = clean_wine_dataset(df)

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = split_data(X, y)

print("Dataset shape after cleaning:", df.shape)
print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Engineered feature present:", "total_acidity" in X.columns)

## 3. Consolidated Test-Set Comparison

The ten algorithm notebooks each write one CSV result into `results/regression/`. The models are ranked by test-set R².

In [ ]:
result_files = [
    "01_linear_regression.csv",
    "02_ridge_regression.csv",
    "03_lasso_regression.csv",
    "04_elasticnet_regression.csv",
    "05_polynomial_regression.csv",
    "06_decision_tree_regressor.csv",
    "07_random_forest_regressor.csv",
    "08_gradient_boosting_regressor.csv",
    "09_svr.csv",
    "10_knn_regressor.csv",
]

result_frames = []
for filename in result_files:
    path = RESULTS_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing regression result: {path}")
    result_frames.append(pd.read_csv(path))

comparison = pd.concat(result_frames, ignore_index=True)

comparison = comparison[["Model", "R2", "RMSE", "MAE"]].sort_values(
    "R2", ascending=False
).reset_index(drop=True)
comparison.insert(0, "Rank", range(1, len(comparison) + 1))

display(comparison.style.format({
    "R2": "{:.4f}",
    "RMSE": "{:.4f}",
    "MAE": "{:.4f}",
}))

comparison_path = RESULTS_DIR / "regression_comparison.csv"
comparison.to_csv(comparison_path, index=False)
print("Saved comparison:", comparison_path)

## 4. Two Best Models — 5-Fold Cross-Validated R²

In [ ]:
model_files = {
    "Linear Regression": "01_linear_regression.joblib",
    "Ridge Regression": "02_ridge_regression.joblib",
    "Lasso Regression": "03_lasso_regression.joblib",
    "ElasticNet Regression": "04_elasticnet_regression.joblib",
    "Polynomial Regression": "05_polynomial_regression.joblib",
    "Decision Tree Regressor": "06_decision_tree_regressor.joblib",
    "Random Forest Regressor": "07_random_forest_regressor.joblib",
    "Gradient Boosting Regressor": "08_gradient_boosting_regressor.joblib",
    "Support Vector Regressor (SVR)": "09_svr.joblib",
    "K-Nearest Neighbors Regressor": "10_knn_regressor.joblib",
}

top_two = comparison.head(2)
cv_rows = []

for model_name in top_two["Model"]:
    model_path = MODELS_DIR / model_files[model_name]
    estimator = joblib.load(model_path)

    scores = cross_val_score(
        estimator,
        X_train,
        y_train,
        cv=5,
        scoring="r2",
        n_jobs=-1,
    )

    cv_rows.append({
        "Model": model_name,
        "CV_R2_Mean": scores.mean(),
        "CV_R2_Std": scores.std(),
        "Fold_1": scores[0],
        "Fold_2": scores[1],
        "Fold_3": scores[2],
        "Fold_4": scores[3],
        "Fold_5": scores[4],
    })

cv_results = pd.DataFrame(cv_rows)
display(cv_results.style.format({
    "CV_R2_Mean": "{:.4f}",
    "CV_R2_Std": "{:.4f}",
    "Fold_1": "{:.4f}",
    "Fold_2": "{:.4f}",
    "Fold_3": "{:.4f}",
    "Fold_4": "{:.4f}",
    "Fold_5": "{:.4f}",
}))

cv_path = RESULTS_DIR / "top_two_cross_validation.csv"
cv_results.to_csv(cv_path, index=False)
print("Saved CV results:", cv_path)

## 5. Best Model — Predicted vs Actual and Residual Plot

In [ ]:
best_name = comparison.iloc[0]["Model"]
best_model = joblib.load(MODELS_DIR / model_files[best_name])
best_pred = best_model.predict(X_test)

best_metrics = regression_metrics(y_test, best_pred)
print("Best model by test-set R²:", best_name)
display(pd.DataFrame([best_metrics], index=[best_name]))

plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=best_pred, alpha=0.65)
lims = [min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())]
plt.plot(lims, lims, linestyle="--", label="Ideal: Actual = Predicted")
plt.title(f"{best_name} — Predicted vs Actual")
plt.xlabel("Actual Quality")
plt.ylabel("Predicted Quality")
plt.legend()
plt.tight_layout()
plt.show()

residuals = y_test - best_pred
plt.figure(figsize=(8, 5))
sns.scatterplot(x=best_pred, y=residuals, alpha=0.65)
plt.axhline(0, linestyle="--", label="Zero Residual")
plt.title(f"{best_name} — Residual Plot")
plt.xlabel("Predicted Quality")
plt.ylabel("Residual (Actual - Predicted)")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Feature Importance for a Tree-Based Model

In [ ]:
tree_names = {
    "Decision Tree Regressor",
    "Random Forest Regressor",
    "Gradient Boosting Regressor",
}

tree_candidates = comparison[comparison["Model"].isin(tree_names)]

if tree_candidates.empty:
    print("No tree-based regression result found.")
else:
    tree_name = tree_candidates.iloc[0]["Model"]
    tree_model = joblib.load(MODELS_DIR / model_files[tree_name])

    preprocessor = tree_model.named_steps["preprocess"]
    estimator = tree_model.named_steps["model"]

    feature_names = preprocessor.get_feature_names_out()
    importance = pd.Series(estimator.feature_importances_, index=feature_names)
    top_importance = importance.sort_values(ascending=False).head(20)

    plt.figure(figsize=(10, 7))
    top_importance.sort_values().plot(kind="barh")
    plt.title(f"{tree_name} — Top 20 Feature Importances")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

## 7. Team Interpretation

Write the team's final interpretation here using the actual outputs from this project.

Discuss the test-set comparison, 5-fold cross-validation, R²/RMSE/MAE, important features, and limitations.